In [1]:
!python embed/download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


### Q1. Embedding a Query

In [2]:
from embed.embedder import Embedder
import numpy as np 

In [3]:
q1 = "How does approximate nearest neighbor search work?"
embedder = Embedder() 
q1_embeddings = embedder.encode(q1) 

print(f"Length embedding vector: {q1_embeddings.shape[0]}")
print(f"The first value of the embedded vector is {q1_embeddings[0]:.2f}")

Length embedding vector: 384
The first value of the embedded vector is -0.02


### Loading the Data

In [4]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
print(len(documents))
print(documents[0]['filename'])
print(documents[0]['content'])

72
01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple

### Q2. Cosine Similarity 

In [6]:
file_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"

document = ""
for doc in documents: 
    if doc['filename'] == file_name: 
        document = doc['content'] 
        break 
if document == "": 
    raise RuntimeError("Document not found.")

print(f"Found document:", end="\n------------------------\n")
print(document)

Found document:
------------------------
# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over every document, so it takes a minute on our dataset.
Keeping everything in memory is fine here, but a larger dataset would
need too much space.

The third problem is brute-force search. For every query we compare the
query vector against every single document. With 1,000 documents this is
fine, probably even faster than anything smarter. But as the dataset
grows past 10,000 or so, it gets slow, and we'll want an approximate
method instead.

What we've done 

In [7]:
document_embedding = embedder.encode(document)
print(f"Document embedding: {document_embedding}")

Document embedding: [-6.69167643e-03 -1.96193967e-02 -1.35708193e-02  2.79713570e-02
  6.41942652e-02  2.62990448e-02  8.73513725e-03  7.13346709e-03
 -4.91385379e-02 -1.85060009e-02 -1.84053811e-02  5.55403037e-02
  1.10243993e-02 -2.03783701e-02 -1.14009126e-01  1.41751132e-02
 -1.42544090e-02  4.95837102e-02  8.09291243e-02  4.17696897e-02
 -2.95922934e-02  4.67177304e-03  4.05626713e-02 -3.90501293e-02
  2.19091567e-03  5.23046040e-02 -1.01323557e-02 -5.98989310e-02
  4.16823523e-02  5.88785129e-04  3.38646839e-02  2.94847675e-02
  1.86943411e-02  1.83957273e-01 -9.16205354e-02  4.11525594e-02
 -5.42344414e-02 -1.79471921e-02 -9.08646905e-02 -1.80611278e-02
 -1.64217023e-02  8.13042594e-03 -5.24593921e-02  6.31449688e-02
  4.37100306e-02  3.84262105e-02 -2.89416808e-02 -6.02209808e-02
  7.83269571e-02 -7.00202511e-02 -1.00836881e-01 -1.33440248e-02
 -1.71584515e-02 -1.22921879e-02  1.05927286e-02  2.81294340e-02
 -9.27830821e-04 -3.08686800e-02  5.78356109e-02  2.53501205e-03
  4.6

In [8]:
cosine_similarity = np.dot(document_embedding, q1_embeddings)
print(f"Cosine similarity between query and document: {cosine_similarity:.2f}")

Cosine similarity between query and document: 0.36


### Q3. Chunking and search by hand 

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Len chunks: {len(chunks)}")

Len chunks: 295


In [10]:
chunks[:5]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [11]:
chunks_embeddings = embedder.encode_batch([chunk['content'] for chunk in chunks])
scores = chunks_embeddings.dot(q1_embeddings)

In [12]:
max_index_score = scores.argmax() 
max_score = scores[max_index_score] 
max_score_file = chunks[max_index_score]['filename']

In [13]:
print(f"The chunk with max score is chunk #{max_index_score}, with a score of ~{max_score:.4f}. The chunk belongs to the file {max_score_file}")

The chunk with max score is chunk #94, with a score of ~0.6489. The chunk belongs to the file 02-vector-search/lessons/07-sqlitesearch-vector.md


#### Q4. Vector search with `minsearch` 

**Note**. For this exercise, I will feed the chunked documents into `minsearch`, rather than the raw documents vector. 

In [14]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(chunks_embeddings, chunks)

In [15]:
q4 = "What metric do we use to evaluate a search engine?"
q4_embeddings = embedder.encode(q4)

In [16]:
search_results = vindex.search(q4_embeddings, num_results=5)
print(f"The filename of the first search result is {search_results[0]['filename']}")

The filename of the first search result is 04-evaluation/lessons/05-search-metrics.md


### Q5. Text Search vs. Vector Search  

In [17]:
from minsearch import Index 
index = Index(text_fields=['content'], keyword_fields=['filename'])
index.fit(chunks)

In [18]:
q5 = "How do I store vectors in PostgreSQL?"
q5_embeddings = embedder.encode(q5)

In [19]:
vector_search_results = vindex.search(q5_embeddings, num_results=5)
text_search_results = index.search(q5, num_results=5)

In [20]:
vector_search_filenames = set([res['filename'] for res in vector_search_results])
text_search_filenames = set([res['filename'] for res in text_search_results])

In [21]:
print(f"Vector search retrieved files: {', '.join(vector_search_filenames)}")
print(f"Text search retrieved files: {', '.join(text_search_filenames)}")

Vector search retrieved files: 03-orchestration/lessons/05-rag.md, 02-vector-search/lessons/08-pgvector.md
Text search retrieved files: 02-vector-search/lessons/01-intro.md, 03-orchestration/lessons/05-rag.md, 02-vector-search/lessons/02-embeddings.md


In [22]:
print(f"File retrieved by vector search, but not by text search: {vector_search_filenames.difference(text_search_filenames)}")

File retrieved by vector search, but not by text search: {'02-vector-search/lessons/08-pgvector.md'}


#### Q6. Hybrid Search 

In [23]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [24]:
q6 = "How do I give the model access to tools?"
q6_embeddings = embedder.encode(q6)

In [25]:
vector_search_results = vindex.search(q6_embeddings)
text_search_results = index.search(q6)

In [26]:
rrf_results = rrf([vector_search_results, text_search_results])

In [27]:
print("Best document content by RRF:")
print()
print(rrf_results[0]['content'])

Best document content by RRF:

 function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always
fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:

```python
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output
```

Look at the output. Instead of a message with the answer, the response
contains a `function_call` entry. The model decided it needs to search
the FAQ before answering. Rather than reply, it asked us to run the
search function first.

Look at the arguments too. The model didn't pass our question
verbatim. It judged the raw question wasn't the best query to search
with. So it rewrote our enrollment question into search keywords like
"enroll late join course".

## Executing the function and sending the result back

The function call contains JSON arguments. We 

In [28]:
print(f"The file of the best ranked document by RRF: {rrf_results[0]['filename']}")

The file of the best ranked document by RRF: 01-agentic-rag/lessons/13-function-calling.md
